# Multi-Agent SDP baseline — frozen split

GPU-only DeBERTa-v3-small embeddings and regularized XGBoost, trained with BugHub source labels. The notebook checks the frozen train/validation/test identity hashes before training. A mismatch stops the run so the baseline cannot silently use different records from the treatment experiment.

Outputs are written below `/kaggle/working/outputs` and embeddings below `/kaggle/working/data/embeddings`; no input files are deleted.

In [ ]:
from pathlib import Path
import hashlib, json, os, time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, roc_auc_score, brier_score_loss, precision_score, recall_score)

assert torch.cuda.is_available(), 'CUDA GPU is required; this notebook never falls back to CPU.'
device = torch.device('cuda')
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory // 2**30, 'GiB')
DATASET = Path('/kaggle/input/datasets/jasirdeen/multi-agent-sdp-canonical/bughubs_canonical.parquet')
assert DATASET.is_file(), f'Attach jasirdeen/multi-agent-sdp-canonical; not found: {DATASET}'
WORK = Path('/kaggle/working')
OUT = WORK / 'outputs' / 'baseline' / 'fixed_split'
EMBED = WORK / 'data' / 'embeddings' / 'fixed_split'
OUT.mkdir(parents=True, exist_ok=True); EMBED.mkdir(parents=True, exist_ok=True)
print('Input:', DATASET, 'bytes:', DATASET.stat().st_size)

In [ ]:
# Match src/baseline/data_splitter.py: same filters, stratification, seed, and split sequence.
columns = ['source','project','issue_id','original_label','model_text']
df = pd.read_parquet(DATASET, columns=columns)
df = df.dropna(subset=['model_text','original_label']).copy()
df['model_text'] = df['model_text'].astype(str).str.strip()
df = df[df['model_text'].str.len() >= 20].copy()
label_map = {'bug': 1, 'non-bug': 0}
df['target'] = df['original_label'].astype(str).str.lower().map(label_map)
if df['target'].isna().any(): raise ValueError('Unmapped labels found')
df['target'] = df['target'].astype('int8')
# Join the exact frozen split assignments; canonical Parquet row order may differ across exports.
KEYS = ['source','project','issue_id']
assignment_files = list(Path('/kaggle/input').rglob('split_assignments.parquet'))
if not assignment_files: raise FileNotFoundError(f'Frozen split dataset is not mounted; input roots: {list(Path("/kaggle/input").iterdir())}')
ASSIGNMENTS = assignment_files[0]
print('Using frozen assignments:', ASSIGNMENTS)
assignments = pd.read_parquet(ASSIGNMENTS, columns=KEYS+['split','split_order'])
df[KEYS] = df[KEYS].astype(str); assignments[KEYS] = assignments[KEYS].astype(str)
if assignments.duplicated(KEYS).any(): raise RuntimeError('Split assignment contains duplicate issue identities')
if len(assignments) != len(df): raise RuntimeError(f'Assignment/canonical row mismatch: {len(assignments)} vs {len(df)}')
parts = {}
for split_name in ['train','validation','test']:
    ids = assignments[assignments['split'].eq(split_name)]
    part = ids.merge(df, on=KEYS, how='left', validate='one_to_one', indicator=True)
    if not part['_merge'].eq('both').all(): raise RuntimeError(f'{split_name} contains identities absent from canonical data')
    parts[split_name] = part.sort_values('split_order').drop(columns=['split','split_order','_merge']).reset_index(drop=True)
train, validation, test = parts['train'], parts['validation'], parts['test']
def identity_sha256(frame):
    payload = frame[['source','project','issue_id']].astype(str).to_csv(index=False, header=False).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()
expected = {
 'train': 'fd610f843e7a3c2d2b3d488d477cb4f13b4e9db50deb2eb3444b4f230cf0424f',
 'validation': '26df89a7c62feb50bf5a4925d1b2946f44bab73dc71f8eb5c2961c49d2df55ff',
 'test': 'cc5b147ddba1b01bf252c0ab2d3b0b596bb3674a80458d8f3976208a2f53b0a1'}
splits = {'train': train, 'validation': validation, 'test': test}
actual = {name: identity_sha256(part) for name, part in splits.items()}
print('rows:', {name: len(part) for name, part in splits.items()})
print('identity hashes:', actual)
if actual != expected: raise RuntimeError('Dataset rows/order do not reproduce the frozen local split; stopping before training.')
(OUT / 'split_manifest.json').write_text(json.dumps({'source':'frozen local split assignments','seed':42,'ratios':{'train':0.70,'validation':0.15,'test':0.15},'rows':{k:len(v) for k,v in splits.items()},'identity_sha256':actual}, indent=2))
print('Frozen split identity verified.')

In [ ]:
%pip install -q 'transformers>=4.36,<5' sentencepiece 'xgboost>=2,<4'
from transformers import AutoModel, AutoTokenizer
from torch.utils.data import DataLoader
MODEL_ID = 'microsoft/deberta-v3-small'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
encoder = AutoModel.from_pretrained(MODEL_ID).to(device).half().eval()
for parameter in encoder.parameters(): parameter.requires_grad_(False)
MAX_LENGTH, BATCH_SIZE = 256, 32
HIDDEN = int(encoder.config.hidden_size)
print('Encoder ready on', device, 'hidden size', HIDDEN)

In [ ]:
def extract_split_embeddings(frame, split_name):
    path = EMBED / f'{split_name}_embeddings.npy'
    state_path = EMBED / f'{split_name}_progress.json'
    if path.exists() and state_path.exists():
        state = json.loads(state_path.read_text())
        if state.get('complete') and state.get('rows') == len(frame) and state.get('model') == MODEL_ID:
            print('Reusing completed cache:', path); return np.load(path, mmap_mode='r')
        if state.get('rows') != len(frame) or state.get('model') != MODEL_ID: path.unlink(); state_path.unlink()
    if path.exists() and state_path.exists():
        state = json.loads(state_path.read_text()); done = int(state.get('rows_done', 0))
        matrix = np.load(path, mmap_mode='r+')
    else:
        matrix = np.lib.format.open_memmap(path, mode='w+', dtype=np.float32, shape=(len(frame), HIDDEN)); done = 0
        state = {'model':MODEL_ID, 'rows':len(frame), 'rows_done':0, 'complete':False}
    texts = frame['model_text'].tolist()
    for start in range(done, len(texts), BATCH_SIZE):
        end = min(start+BATCH_SIZE, len(texts))
        batch = tokenizer(texts[start:end], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
        batch = {key: value.to(device) for key, value in batch.items()}
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
            vec = encoder(**batch).last_hidden_state[:,0,:].float().cpu().numpy()
        matrix[start:end] = vec
        if end % (BATCH_SIZE*100) == 0 or end == len(texts):
            matrix.flush(); state['rows_done'] = end; state_path.write_text(json.dumps(state))
            print(f'{split_name}: {end:,}/{len(texts):,}')
    matrix.flush(); state.update({'rows_done':len(texts), 'complete':True}); state_path.write_text(json.dumps(state))
    return np.load(path, mmap_mode='r')

X_train = extract_split_embeddings(train, 'train')
X_validation = extract_split_embeddings(validation, 'validation')
X_test = extract_split_embeddings(test, 'test')
print('embedding shapes:', X_train.shape, X_validation.shape, X_test.shape)

In [ ]:
import xgboost as xgb
xgb_model = xgb.XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.7, colsample_bytree=0.6, reg_alpha=2.0, reg_lambda=10.0, gamma=1.0,
    min_child_weight=3.0, scale_pos_weight=1.0, eval_metric='logloss',
    early_stopping_rounds=35, tree_method='hist', device='cuda', random_state=42)
y_train = train['target'].to_numpy(); y_val = validation['target'].to_numpy(); y_test = test['target'].to_numpy()
xgb_model.fit(X_train, y_train, eval_set=[(X_train,y_train),(X_validation,y_val)], verbose=50)
val_probability = xgb_model.predict_proba(X_validation)[:,1]
thresholds = np.arange(0.40, 0.70, 0.01)
threshold = round(float(max(thresholds, key=lambda t: accuracy_score(y_val, (val_probability >= t).astype(int)))),2)
test_probability = xgb_model.predict_proba(X_test)[:,1]
test_prediction = (test_probability >= threshold).astype('int8')
metrics = {
 'accuracy':float(accuracy_score(y_test,test_prediction)),
 'balanced_accuracy':float(balanced_accuracy_score(y_test,test_prediction)),
 'precision_bug':float(precision_score(y_test,test_prediction,pos_label=1,zero_division=0)),
 'recall_bug':float(recall_score(y_test,test_prediction,pos_label=1,zero_division=0)),
 'f1_bug':float(f1_score(y_test,test_prediction,pos_label=1,zero_division=0)),
 'macro_f1':float(f1_score(y_test,test_prediction,average='macro',zero_division=0)),
 'roc_auc':float(roc_auc_score(y_test,test_probability)),
 'matthews_corrcoef':float(matthews_corrcoef(y_test,test_prediction)),
 'brier_score':float(brier_score_loss(y_test,test_probability)),
 'optimal_threshold':threshold, 'train_records':len(train), 'validation_records':len(validation), 'test_records':len(test),
 'split_seed':42, 'model_id':MODEL_ID, 'test_identity_sha256':actual['test'],
 'train_identity_sha256':actual['train'], 'validation_identity_sha256':actual['validation']}
model_path = OUT / 'xgb_baseline.json'; xgb_model.save_model(model_path)
(OUT / 'threshold_meta.json').write_text(json.dumps({'optimal_threshold':threshold},indent=2))
(OUT / 'metrics.json').write_text(json.dumps(metrics,indent=2))
pd.DataFrame({'source':test['source'],'project':test['project'],'issue_id':test['issue_id'],
              'original_label':test['original_label'],'target':y_test,
              'probability_bug':test_probability,'prediction':test_prediction}).to_parquet(OUT/'test_predictions.parquet',index=False)
print(json.dumps(metrics,indent=2)); print('Saved model and metrics to', OUT)